In [2]:
import DB
import pandas as pd
import numpy as np
import os
import json
import datetime
from pymongo import MongoClient
from dotenv import load_dotenv
from tqdm import tqdm
from pprint import pprint
from collections import defaultdict

In [3]:
%load_ext autoreload
%autoreload 2

# 모델 설계

In [72]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from scipy.special import expit
import xgboost as xgb

In [38]:
db = DB.Load()
collection = db.get_collection('duo_matches_4')
duo_matches_cursor = collection.find({},
            {'duo_key':1, '_id':0, 'matches': 1}
        )
duo_matches = [{'duo_key': item['duo_key'], 'matchIds': item['matches']} for item in duo_matches_cursor]

In [39]:
duo_matches

[{'duo_key': 'Path Integral#Korea@김다미 귀욤이#KR1',
  'matchIds': ['KR_7447794766',
   'KR_7447835513',
   'KR_7506637533',
   'KR_7506691067']},
 {'duo_key': 'BOOR#KR1@김다미 귀욤이#KR1',
  'matchIds': ['KR_7463636618',
   'KR_7463452213',
   'KR_7463500658',
   'KR_7463320714',
   'KR_7463420202',
   'KR_7463561824',
   'KR_7463385506']},
 {'duo_key': '김다미 귀욤이#KR1@대가리개뿌사뿐다#KR1',
  'matchIds': ['KR_7463636618',
   'KR_7463452213',
   'KR_7463500658',
   'KR_7463420202',
   'KR_7463561824',
   'KR_7463385506']},
 {'duo_key': '닉변하지마본주다#KR1@행단보도#KR1',
  'matchIds': ['KR_7480393256',
   'KR_7481713923',
   'KR_7480197622',
   'KR_7480023637',
   'KR_7479766745',
   'KR_7480352036',
   'KR_7480321695',
   'KR_7481729954',
   'KR_7480138959',
   'KR_7480230739',
   'KR_7479957036',
   'KR_7479876258',
   'KR_7480299851',
   'KR_7480075833',
   'KR_7476646598',
   'KR_7480271845']},
 {'duo_key': 'IlllIlIIllI#101@닉변하지마본주다#KR1',
  'matchIds': ['KR_7475355250',
   'KR_7475388018',
   'KR_7475402132',
   

In [44]:
with open('./all_avg_data.json', 'r', encoding = 'UTF8') as avg_data:
    data = json.load(avg_data)

In [45]:
data

[{'summonerName': 'Path Integral',
  'summonerTag': 'Korea',
  'killParticipation': 0.4696050272016056,
  'kda': 3.500757575757576,
  'visionScore': 26.7,
  'winRate': 50.0},
 {'summonerName': '김다미 귀욤이',
  'summonerTag': 'KR1',
  'killParticipation': 0.5092614906444464,
  'kda': 2.599285714285714,
  'visionScore': 34.15,
  'winRate': 65.0},
 {'summonerName': 'BOOR',
  'summonerTag': 'KR1',
  'killParticipation': 0.5341186623469787,
  'kda': 2.947916666666667,
  'visionScore': 43.65,
  'winRate': 30.0},
 {'summonerName': '대가리개뿌사뿐다',
  'summonerTag': 'KR1',
  'killParticipation': 0.3671207339169317,
  'kda': 2.6788131313131314,
  'visionScore': 20.8,
  'winRate': 55.00000000000001},
 {'summonerName': '닉변하지마본주다',
  'summonerTag': 'KR1',
  'killParticipation': 0.5289516182408508,
  'kda': 4.2672601010101,
  'visionScore': 33.25,
  'winRate': 55.00000000000001},
 {'summonerName': 'IlllIlIIllI',
  'summonerTag': '101',
  'killParticipation': 0.5586612300824568,
  'kda': 4.4476556776556775,
 

In [55]:
all_data = []
y = []
pbar = tqdm(duo_matches,
           total = len(duo_matches))
for duo in pbar:
    players = duo['duo_key'].split('@')
    player1gameName, player1tagLine = players[0].split('#')
    player2gameName, player2tagLine = players[1].split('#')
    data_duo = {'p1KP': 0, 'p1KDA': 0, 'p1VS': 0, 'p1WR': 0, 'p2KP': 0, 'p2KDA': 0, 'p2VS': 0, 'p2WR': 0}

    p1ready = False
    p2ready = False
    for summoner in data:
        if (summoner['summonerName'] == player1gameName) and (summoner['summonerTag'] == player1tagLine):
            data_duo['p1KP'], data_duo['p1KDA'], data_duo['p1VS'], data_duo['p1WR'] = summoner['killParticipation'], summoner['kda'], summoner['visionScore'], summoner['winRate']
            p1ready = True
        elif (summoner['summonerName'] == player2gameName) and (summoner['summonerTag'] == player2tagLine):
            data_duo['p2KP'], data_duo['p2KDA'], data_duo['p2VS'], data_duo['p2WR'] = summoner['killParticipation'], summoner['kda'], summoner['visionScore'], summoner['winRate']
            p2ready = True

        if p1ready & p2ready:
            break
    all_data.append(data_duo)

    duo_winrate_sum = 0
    for matchId in duo['matchIds']:
        for participant in db.find_match(int(matchId.split('_')[1]))['participants']:
            if participant['riotIdGameName']==player1gameName:
                if participant['win']:
                    duo_winrate_sum += 1
                break
    y.append(duo_winrate_sum/len(duo['matchIds']))


00%|████████████████████████████████████████████████████████████████████████████████| 291/291 [00:29<00:00,  9.81it/s]

In [57]:
y

[0.5,
 0.42857142857142855,
 0.5,
 0.625,
 0.5714285714285714,
 0.5,
 0.16666666666666666,
 0.29411764705882354,
 0.75,
 0.7692307692307693,
 0.7,
 0.5,
 0.5714285714285714,
 0.5,
 0.6,
 0.75,
 0.75,
 0.14285714285714285,
 0.3333333333333333,
 0.16666666666666666,
 0.0,
 0.5384615384615384,
 0.3333333333333333,
 0.5,
 0.5,
 0.125,
 0.24,
 0.2,
 0.631578947368421,
 0.0,
 0.3333333333333333,
 0.5862068965517241,
 0.6190476190476191,
 1.0,
 0.0,
 0.0,
 0.0,
 1.0,
 1.0,
 1.0,
 1.0,
 0.5,
 0.5238095238095238,
 0.42857142857142855,
 0.5,
 0.5,
 0.5454545454545454,
 0.6086956521739131,
 0.8333333333333334,
 0.3333333333333333,
 0.3333333333333333,
 0.5,
 0.8333333333333334,
 0.5714285714285714,
 0.8,
 0.5806451612903226,
 0.5,
 0.5,
 0.5,
 0.6,
 0.25,
 0.375,
 0.75,
 0.625,
 0.5,
 0.5,
 0.5714285714285714,
 0.6153846153846154,
 0.5714285714285714,
 0.625,
 0.5,
 0.4166666666666667,
 0.5,
 0.3333333333333333,
 0.75,
 0.6153846153846154,
 0.6451612903225806,
 0.391304347826087,
 0.5294117647058

In [59]:
X = np.array([[duo['p1KP'], duo['p1KDA'], duo['p1VS'], duo['p1WR'],
               duo['p2KP'], duo['p2KDA'], duo['p2VS'], duo['p2WR']] for duo in all_data])
y = np.array(y)

In [68]:
# 시야점수에 대한 비선형 변환 (로그 변환)
X[:, 2] = np.log1p(X[:, 2])  # p1VS - log 변환 (시야점수에 대한 변환)
X[:, 6] = np.log1p(X[:, 6])  # p2VS - log 변환 (시야점수에 대한 변환)

# KDA 값에 시그모이드 함수 적용
X[:, 1] = np.array([expit(kda - 2) for kda in X[:, 1]])  # p1KDA 변환
X[:, 5] = np.array([expit(kda - 2) for kda in X[:, 5]])  # p2KDA 변환

# 킬 관여도에 대한 비선형 변환 (이차 변환 적용)
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X[:, [0, 4]])  # p1KP, p2KP에 대해 이차 변환

# 변환된 특성에 추가
X = np.hstack([X, X_poly[:, 2:]])  # p1KP, p2KP에 대한 이차항 추가

# 특성 스케일링 (전체 데이터 스케일링)
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# 훈련 데이터와 테스트 데이터 분리 (80% 훈련, 20% 테스트)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [71]:
from sklearn.model_selection import KFold, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb
from sklearn.metrics import mean_squared_error

In [70]:
# KFold 설정 (5-Fold 교차 검증)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 모델 정의 (여러 가지 모델)
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
    "XGBoost": xgb.XGBRegressor(n_estimators=100, random_state=42)
}

# 각 모델에 대해 K-Fold 교차 검증을 실행
for model_name, model in models.items():
    print(f"Running {model_name} with K-Fold Cross Validation")
    
    # K-Fold 교차 검증
    cv_results = cross_val_score(model, X_scaled, y, cv=kf, scoring='neg_mean_squared_error')
    
    # MSE 평균 출력
    print(f'{model_name} - MSE (Negative): {-cv_results.mean()}')
    print(f'{model_name} - Standard Deviation: {cv_results.std()}\n')

Running Linear Regression with K-Fold Cross Validation
Linear Regression - MSE (Negative): 0.06472452996289639
Linear Regression - Standard Deviation: 0.007332899935676423

Running Random Forest with K-Fold Cross Validation
Random Forest - MSE (Negative): 0.056790867630091
Random Forest - Standard Deviation: 0.005021536089649396

Running Gradient Boosting with K-Fold Cross Validation
Gradient Boosting - MSE (Negative): 0.05803736509751097
Gradient Boosting - Standard Deviation: 0.005588662862502011

Running XGBoost with K-Fold Cross Validation
XGBoost - MSE (Negative): 0.05917372208648817
XGBoost - Standard Deviation: 0.005104560028137422



## feature 추가

In [73]:
# 각 특성에 대해 평균과 차이 계산
p1KP, p1KDA, p1VS, p1WR = X[:, 0], X[:, 1], X[:, 2], X[:, 3]
p2KP, p2KDA, p2VS, p2WR = X[:, 4], X[:, 5], X[:, 6], X[:, 7]

# 평균과 차이 계산
avg_KP = (p1KP + p2KP) / 2
diff_KP = np.abs(p1KP - p2KP)
avg_KDA = (p1KDA + p2KDA) / 2
diff_KDA = np.abs(p1KDA - p2KDA)
avg_VS = (p1VS + p2VS) / 2
diff_VS = np.abs(p1VS - p2VS)
avg_WR = (p1WR + p2WR) / 2
diff_WR = np.abs(p1WR - p2WR)

# 새로운 특성 추가
X = np.hstack([X, avg_KP.reshape(-1, 1), diff_KP.reshape(-1, 1), avg_KDA.reshape(-1, 1), diff_KDA.reshape(-1, 1),
               avg_VS.reshape(-1, 1), diff_VS.reshape(-1, 1), avg_WR.reshape(-1, 1), diff_WR.reshape(-1, 1)])


In [74]:
# 시야점수에 대한 비선형 변환 (로그 변환)
X[:, 2] = np.log1p(X[:, 2])  # p1VS - log 변환 (시야점수에 대한 변환)
X[:, 6] = np.log1p(X[:, 6])  # p2VS - log 변환 (시야점수에 대한 변환)

# KDA 값에 시그모이드 함수 적용
X[:, 1] = np.array([expit(kda - 2) for kda in X[:, 1]])  # p1KDA 변환
X[:, 5] = np.array([expit(kda - 2) for kda in X[:, 5]])  # p2KDA 변환

# 킬 관여도에 대한 비선형 변환 (이차 변환 적용)
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X[:, [0, 4]])  # p1KP, p2KP에 대해 이차 변환

# 변환된 특성에 추가
X = np.hstack([X, X_poly[:, 2:]])  # p1KP, p2KP에 대한 이차항 추가

# 특성 스케일링 (전체 데이터 스케일링)
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# 훈련 데이터와 테스트 데이터 분리 (80% 훈련, 20% 테스트)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)


In [75]:
# KFold 설정 (5-Fold 교차 검증)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 모델 정의 (여러 가지 모델)
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
    "XGBoost": xgb.XGBRegressor(n_estimators=100, random_state=42)
}

# 그리드 서치를 위한 하이퍼파라미터 설정
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

param_grid_gb = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7]
}

param_grid_xgb = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0]
}

# 그리드 서치를 통해 하이퍼파라미터 튜닝
grid_search_rf = GridSearchCV(RandomForestRegressor(random_state=42), param_grid_rf, cv=kf, scoring='neg_mean_squared_error')
grid_search_gb = GridSearchCV(GradientBoostingRegressor(random_state=42), param_grid_gb, cv=kf, scoring='neg_mean_squared_error')
grid_search_xgb = GridSearchCV(xgb.XGBRegressor(random_state=42), param_grid_xgb, cv=kf, scoring='neg_mean_squared_error')

# 모델 튜닝
grid_search_rf.fit(X_train, y_train)
grid_search_gb.fit(X_train, y_train)
grid_search_xgb.fit(X_train, y_train)

# 결과 출력
print("Best parameters for Random Forest: ", grid_search_rf.best_params_)
print("Best parameters for Gradient Boosting: ", grid_search_gb.best_params_)
print("Best parameters for XGBoost: ", grid_search_xgb.best_params_)

Best parameters for Random Forest:  {'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}
Best parameters for Gradient Boosting:  {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 50}
Best parameters for XGBoost:  {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 50, 'subsample': 0.8}


In [76]:
# 각 모델에 대해 K-Fold 교차 검증을 실행
for model_name, model in models.items():
    print(f"Running {model_name} with K-Fold Cross Validation")
    
    # K-Fold 교차 검증
    cv_results = cross_val_score(model, X_scaled, y, cv=kf, scoring='neg_mean_squared_error')
    
    # MSE 평균 출력
    print(f'{model_name} - MSE (Negative): {-cv_results.mean()}')
    print(f'{model_name} - Standard Deviation: {cv_results.std()}\n')

Running Linear Regression with K-Fold Cross Validation
Linear Regression - MSE (Negative): 0.06823271512157167
Linear Regression - Standard Deviation: 0.007110297640901919

Running Random Forest with K-Fold Cross Validation
Random Forest - MSE (Negative): 0.05831351912569773
Random Forest - Standard Deviation: 0.005103582294150701

Running Gradient Boosting with K-Fold Cross Validation
Gradient Boosting - MSE (Negative): 0.05925378164092103
Gradient Boosting - Standard Deviation: 0.005339333510449961

Running XGBoost with K-Fold Cross Validation
XGBoost - MSE (Negative): 0.06242491388659245
XGBoost - Standard Deviation: 0.004151374881102083



In [77]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [80]:
def evaluation(best_model_name):
    # 선택한 모델을 다시 학습
    best_model = models[best_model_name]
    
    # 하이퍼파라미터 튜닝된 모델 사용
    if best_model_name == "Random Forest":
        best_model = grid_search_rf.best_estimator_
    elif best_model_name == "Gradient Boosting":
        best_model = grid_search_gb.best_estimator_
    elif best_model_name == "XGBoost":
        best_model = grid_search_xgb.best_estimator_
    else:
        best_model = models[best_model_name]
    
    # 모델 훈련
    best_model.fit(X_train, y_train)
    
    # 테스트 데이터로 예측
    y_pred = best_model.predict(X_test)
    
    # MSE (Mean Squared Error)
    mse = mean_squared_error(y_test, y_pred)
    print(f'{best_model_name} - MSE: {mse}')
    
    # RMSE (Root Mean Squared Error)
    rmse = np.sqrt(mse)
    print(f'{best_model_name} - RMSE: {rmse}')
    
    # MAE (Mean Absolute Error)
    mae = mean_absolute_error(y_test, y_pred)
    print(f'{best_model_name} - MAE: {mae}')
    
    # R^2 (R-squared)
    r2 = r2_score(y_test, y_pred)
    print(f'{best_model_name} - R^2: {r2}')

In [81]:
evaluation("Random Forest")

Random Forest - MSE: 0.057637338874332825
Random Forest - RMSE: 0.2400777767189892
Random Forest - MAE: 0.17151282056621875
Random Forest - R^2: -0.19667345637463862


In [82]:
evaluation("Gradient Boosting")

Gradient Boosting - MSE: 0.05704617668034572
Gradient Boosting - RMSE: 0.238843414563487
Gradient Boosting - MAE: 0.17872736871217615
Gradient Boosting - R^2: -0.18439967483349262


In [83]:
evaluation("XGBoost")

XGBoost - MSE: 0.05651084716378003
XGBoost - RMSE: 0.23772010256555928
XGBoost - MAE: 0.17570202633196397
XGBoost - R^2: -0.17328509814762638


상관계수가 음수가 나와버리는 기염을 토했습니다!

# 임의로 값 지정

In [84]:
db = DB.Load()

In [124]:
def Score(player1GameName, player1TagLine, player2GameName, player2TagLine):
    # 최대 300게임 정보 로드
    
    UpdateDB(player1GameName, player1TagLine) # 각 플레이어 정보 최신화
    UpdateDB(player2GameName, player2TagLine) 
    
    player1Data = db.find_summoner(player1GameName, player1TagLine)
    player2Data = db.find_summoner(player2GameName, player2TagLine)

    if (player1Data == None) or (player2Data == None):
        return 101
    elif (len(player1Data['match_ids']) < 50) or (len(player2Data['match_ids']) < 50):
        return 102
    elif np.abs(DB.tiers.index(player1Data['tier']) - DB.tiers.index(player2Data['tier'])) > 2:
        return 103
    
    # 정보 가공
    # 시야점수
    
    # KDA
    
    
    # 라인

    
    # 킬 관여도
    
    
    # 출력: (int) 0~100 점수, 101 = id not found, 102 = lack of information, 103 = invalid rank, 104 = 연산오류
    return score
    

In [125]:
def UpdateDB(gameName, tagLine):
    last_update = db.find_summoner(gameName, tagLine)
    if last_update == None:
        # 100게임 모두 수집
        print('100게임 모두 수집')
    else:
        last_update = last_update['updated_at']
        # gameEndTimestamp가 last_update보다 예전인 경우가 나올 때까지 매치데이터 수집
        print('100게임 matchid 수집')
        print('100게임 반복문 돌리기')
        # 300게임 이상의 데이터 절삭

In [126]:
Score('BOOR','KR1','김다미 귀욤이','KR1')

100게임 모두 수집
100게임 matchid 수집
100게임 반복문 돌리기


101

In [128]:
Score('Path Integral','Korea','김다미 귀욤이','KR1')

100게임 모두 수집
100게임 matchid 수집
100게임 반복문 돌리기


101